# Notebook 02: grad, vmap, and jit -- JAX's Three Power Transforms

## Why This Matters

JAX's three core transforms -- `grad`, `vmap`, and `jit` -- are what separate it from "just fast NumPy." Together they let you write research-quality code that is simultaneously correct, vectorized, and compiled. Per-sample gradients (needed for differential privacy, second-order methods, and meta-learning) are trivial in JAX via `vmap(grad(...))` but notoriously painful in PyTorch. Understanding how to compose these transforms is the single most important JAX skill.

## Background

All three transforms work by **function transformation** -- they take a function as input and return a new function. This is fundamentally different from PyTorch's approach of imperative computation with a hidden gradient tape.

- `grad(f)` returns a function that computes df/dx
- `vmap(f)` returns a function that maps f over a batch dimension without for-loops
- `jit(f)` returns an XLA-compiled version of f

They compose freely: `jit(vmap(grad(f)))` is valid and efficient.

In [ ]:
%pip install -q jax jaxlib numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time

jax.config.update("jax_enable_x64", True)
print("JAX version:", jax.__version__)
print("Backend:", jax.default_backend())
print("Ready.")

## 1. jax.grad: Functional Automatic Differentiation

In PyTorch you call `loss.backward()` which populates `.grad` attributes. JAX is different: `jax.grad(f)` **returns a new function** that computes the gradient of `f` with respect to its first argument.

$$\nabla_x f(x) = \frac{\partial f}{\partial x}$$

Key properties:
- `grad(f)` returns a **function**, not a value
- By default, differentiates with respect to the **first argument**
- The output must be a **scalar** (for vector-valued functions, use `jacobian`)
- Use `argnums` to differentiate w.r.t. other arguments

In [ ]:
# Basic grad usage
def f(x):
    return jnp.sum(x ** 2)  # f(x) = sum(x_i^2), gradient = 2x

grad_f = jax.grad(f)

x = jnp.array([1.0, 2.0, 3.0])
print("f(x) =", f(x))
print("grad(f)(x) =", grad_f(x))  # should be [2, 4, 6]
print("Expected:   [2. 4. 6.]")

# Differentiating w.r.t. a specific argument
def loss(params, x, y):
    w, b = params
    pred = w * x + b
    return jnp.mean((pred - y) ** 2)

grad_loss = jax.grad(loss, argnums=0)

params = (jnp.array(2.0), jnp.array(0.5))
x_data = jnp.array([1.0, 2.0, 3.0])
y_data = jnp.array([3.0, 5.0, 7.0])  # y = 2x + 1

grads = grad_loss(params, x_data, y_data)
print("\nLinear regression gradients:")
print("  dL/dw =", grads[0])
print("  dL/db =", grads[1])

manual_dw = 2 * jnp.mean((params[0]*x_data + params[1] - y_data) * x_data)
print("  Manual dL/dw =", manual_dw)


## 2. jax.value_and_grad: Getting Value and Gradient Together

When training, you need both the loss value (for logging) and its gradient (for updates). Computing them separately would require two forward passes. `value_and_grad` computes both in one pass.

In [ ]:
# value_and_grad: compute loss and gradient simultaneously
def cross_entropy(logits, labels):
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(len(labels)), labels])

value_and_grad_fn = jax.value_and_grad(cross_entropy)

key = jax.random.PRNGKey(0)
logits = jax.random.normal(key, (8, 10))  # batch=8, classes=10
labels = jax.random.randint(key, (8,), 0, 10)

loss_val, grads = value_and_grad_fn(logits, labels)
print("Loss:", loss_val)
print("Gradient shape:", grads.shape)

# Practical example: differentiating w.r.t. model params
def model_loss(params, x, y):
    W, b = params
    logits = x @ W + b
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(len(y)), y])

W = jax.random.normal(key, (4, 3))
b = jnp.zeros(3)
x_batch = jax.random.normal(key, (8, 4))
y_batch = jax.random.randint(key, (8,), 0, 3)

loss_and_grad = jax.value_and_grad(model_loss)
loss_val, (dW, db) = loss_and_grad((W, b), x_batch, y_batch)
print("\nModel loss:", loss_val)
print("dW shape:", dW.shape, "db shape:", db.shape)


## 3. Higher-Order Gradients: The Hessian is Just grad(grad(f))

Higher-order differentiation is just function composition. The Hessian (matrix of second derivatives) is `jax.hessian(f)`, internally implemented as `jacfwd(jacrev(f))`.

$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$

This enables second-order optimization (Newton's method), curvature analysis, and Neural Tangent Kernel computation.

In [ ]:
# Higher-order gradients
def quadratic(x):
    A = jnp.array([[3.0, 1.0], [1.0, 2.0]])
    return x @ A @ x

x = jnp.array([1.0, 2.0])

grad_f = jax.grad(quadratic)
print("grad(f)(x):", grad_f(x))

hessian_f = jax.hessian(quadratic)
H = hessian_f(x)
print("Hessian:\n", H)
print("Expected 2*A =\n", 2 * jnp.array([[3.0, 1.0], [1.0, 2.0]]))

eigenvalues = jnp.linalg.eigvalsh(H)
print("Eigenvalues:", eigenvalues)
print("Convex:", jnp.all(eigenvalues > 0))

# Newton step: x_new = x - H^{-1} grad(f)(x)
grad_val = grad_f(x)
newton_step = jnp.linalg.solve(H, grad_val)
x_min = x - newton_step
print("Minimum via Newton:", x_min, "  f(x_min):", quadratic(x_min))


## 4. jax.vmap: Vectorize Without Writing Batch Loops

`vmap` (vectorized map) takes a function written for a single example and automatically maps it over a batch dimension. **You write code for one example, vmap handles batching**.

This is not just syntactic sugar -- vmap generates efficient batched XLA code, not a Python for-loop.

In [ ]:
# vmap: single-example function -> batched function
def single_loss(w, x, y):
    pred = jnp.dot(w, x)
    return (pred - y) ** 2

# in_axes=(None, 0, 0): w is shared, x and y are batched
batched_loss = jax.vmap(single_loss, in_axes=(None, 0, 0))

key = jax.random.PRNGKey(0)
w = jnp.ones(4)
X_batch = jax.random.normal(key, (8, 4))
Y_batch = jax.random.normal(key, (8,))

per_sample_losses = batched_loss(w, X_batch, Y_batch)
print("Per-sample losses shape:", per_sample_losses.shape)
print("Per-sample losses:", per_sample_losses)

# Verify correctness
loop_losses = jnp.array([single_loss(w, X_batch[i], Y_batch[i]) for i in range(8)])
print("Match with loop:", jnp.allclose(per_sample_losses, loop_losses))

# Performance comparison
@jax.jit
def batched_vmap_fn(w, X, Y):
    return jax.vmap(single_loss, in_axes=(None, 0, 0))(w, X, Y)

X_large = jax.random.normal(key, (1000, 64))
Y_large = jax.random.normal(key, (1000,))
w_large = jnp.ones(64)

_ = batched_vmap_fn(w_large, X_large, Y_large).block_until_ready()  # warmup

t0 = time.perf_counter()
for _ in range(100):
    _ = batched_vmap_fn(w_large, X_large, Y_large).block_until_ready()
vmap_time = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(5):
    _ = jnp.array([single_loss(w_large, X_large[i], Y_large[i]) for i in range(1000)])
loop_time = (time.perf_counter() - t0) / 5 * 1000

print(f"\nvmap: {vmap_time:.2f} ms | loop: {loop_time:.2f} ms | speedup: {loop_time/vmap_time:.1f}x")


## 5. Per-Sample Gradients: vmap(grad(...)) -- Trivial in JAX

**Per-sample gradients** (gradient of loss_i w.r.t. params for each training example) are required for:
- **DP-SGD** (differential private training clips per-sample gradients before averaging)
- **Influence functions** (measuring how each training point affects predictions)
- **MAML** inner loops (per-task gradients in meta-learning)

In JAX: `vmap(grad(loss_fn))`. One line.

In [ ]:
# Per-sample gradients via vmap(grad(...))
def per_sample_bce(params, x, y):
    W, b = params
    logit = jnp.dot(W, x) + b
    return -(y * jax.nn.log_sigmoid(logit) + (1 - y) * jax.nn.log_sigmoid(-logit))

grad_single = jax.grad(per_sample_bce)
per_sample_grads = jax.vmap(grad_single, in_axes=(None, 0, 0))

key = jax.random.PRNGKey(42)
batch_size, input_dim = 16, 8
W = jax.random.normal(key, (input_dim,)) * 0.1
b = jnp.zeros(())
params = (W, b)

X = jax.random.normal(key, (batch_size, input_dim))
Y = jax.random.bernoulli(key, p=0.5, shape=(batch_size,)).astype(jnp.float32)

grads_W, grads_b = per_sample_grads(params, X, Y)
print("Per-sample dL/dW shape:", grads_W.shape)  # (16, 8)
print("Per-sample dL/db shape:", grads_b.shape)  # (16,)

# DP-SGD: clip per-sample gradients to max norm C, then average
clip_norm = 1.0

def clip_grad_vec(g):
    norm = jnp.linalg.norm(g)
    return g * jnp.minimum(1.0, clip_norm / (norm + 1e-8))

clipped_W = jax.vmap(clip_grad_vec)(grads_W)
mean_clipped_W = jnp.mean(clipped_W, axis=0)

print("\nDP-SGD clipping:")
print("  Raw grad norm:     ", jnp.linalg.norm(jnp.mean(grads_W, axis=0)))
print("  Clipped grad norm: ", jnp.linalg.norm(mean_clipped_W))
norms_before = jnp.linalg.norm(grads_W, axis=1)
norms_after  = jnp.linalg.norm(clipped_W, axis=1)
print("  Max per-sample norm before clip:", norms_before.max())
print("  Max per-sample norm after clip: ", norms_after.max())


## 6. jax.jit: XLA Compilation

`jit` compiles a function using XLA. Rules:
1. **No Python control flow on traced JAX values** -- use `jax.lax.cond` instead of `if`
2. **No data-dependent shapes** -- `x[x > 0]` fails (shape unknown at compile time)
3. **Use `jax.debug.print` for runtime printing**, not Python `print`
4. **Static arguments** -- use `static_argnums` for Python values JAX should not trace

In [ ]:
# jit benchmark
@jax.jit
def relu_network(x, W1, b1, W2, b2):
    h = jax.nn.relu(x @ W1 + b1)
    return h @ W2 + b2

def relu_network_nojit(x, W1, b1, W2, b2):
    h = jax.nn.relu(x @ W1 + b1)
    return h @ W2 + b2

key = jax.random.PRNGKey(0)
x    = jax.random.normal(key, (32, 128))
W1   = jax.random.normal(key, (128, 256)) * 0.01
b1   = jnp.zeros(256)
W2   = jax.random.normal(key, (256, 10)) * 0.01
b2   = jnp.zeros(10)

_ = relu_network(x, W1, b1, W2, b2).block_until_ready()  # warmup / compile

t0 = time.perf_counter()
for _ in range(500):
    _ = relu_network(x, W1, b1, W2, b2).block_until_ready()
jit_time = (time.perf_counter() - t0) / 500 * 1000

t0 = time.perf_counter()
for _ in range(500):
    _ = relu_network_nojit(x, W1, b1, W2, b2).block_until_ready()
nojit_time = (time.perf_counter() - t0) / 500 * 1000

print(f"JIT:    {jit_time:.4f} ms/call")
print(f"No JIT: {nojit_time:.4f} ms/call")
print(f"Speedup: {nojit_time/jit_time:.2f}x")

# static_argnums example
act_fn = jax.jit(
    lambda x, act: jax.nn.relu(x) if act == "relu" else jnp.tanh(x),
    static_argnums=(1,)
)
x_test = jnp.array([-1.0, 0.0, 1.0, 2.0])
print("\nrelu:", act_fn(x_test, "relu"))
print("tanh: ", act_fn(x_test, "tanh"))


## 7. The JAX Composition Pattern: jit(vmap(grad(f)))

Composing all three transforms gives you per-sample gradients, efficiently computed with XLA. This pattern appears in DP-SGD, MAML, NTK computation, and physics-informed neural networks.

In [ ]:
# Full composition: jit(vmap(value_and_grad(f)))
def single_xe_loss(params, x, y):
    W1, b1, W2, b2 = params
    h = jax.nn.relu(x @ W1 + b1)
    logits = h @ W2 + b2
    log_probs = jax.nn.log_softmax(logits)
    return -log_probs[y]

per_sample_val_grad = jax.jit(
    jax.vmap(jax.value_and_grad(single_xe_loss), in_axes=(None, 0, 0))
)

key = jax.random.PRNGKey(0)
k1, k2, k3, k4 = jax.random.split(key, 4)
params = (
    jax.random.normal(k1, (16, 32)) * 0.1,
    jnp.zeros(32),
    jax.random.normal(k2, (32, 5)) * 0.1,
    jnp.zeros(5),
)
X = jax.random.normal(k3, (64, 16))
Y = jax.random.randint(k4, (64,), 0, 5)

losses, (dW1, db1, dW2, db2) = per_sample_val_grad(params, X, Y)

print("Composition: jit(vmap(value_and_grad(loss)))")
print("  per-sample losses shape:", losses.shape)
print("  per-sample dW1 shape:   ", dW1.shape)
print("  mean loss:", losses.mean())

# Mean gradient = standard batch gradient
mean_dW1 = dW1.mean(axis=0)
print("  mean dW1 shape:", mean_dW1.shape)

# Timing
_ = per_sample_val_grad(params, X, Y)  # warmup
t0 = time.perf_counter()
for _ in range(200):
    _ = per_sample_val_grad(params, X, Y)[0].block_until_ready()
t = (time.perf_counter() - t0) / 200 * 1000
print(f"\njit(vmap(value_and_grad)): {t:.3f} ms for batch=64")


## Summary

### Key Concepts

| Transform | What it does | Returns | Common Pitfall |
|-----------|-------------|---------|----------------|
| `jax.grad(f)` | Differentiate f w.r.t. first arg | A function | f must return a scalar |
| `jax.value_and_grad(f)` | Both value and gradient | A function returning (val, grad) | Don't compute separately |
| `jax.hessian(f)` | Full Hessian matrix | A function | Expensive -- O(n^2) entries |
| `jax.vmap(f)` | Vectorize over batch dim | A function | Check `in_axes` carefully |
| `jax.jit(f)` | XLA-compile f | A faster function | Python side effects break silently |

**The key insight**: `jit(vmap(grad(f)))` computes per-sample gradients with one line of code -- the single pattern that makes JAX the preferred framework for differential privacy research.

**Next**: `03_lax_and_control_flow.ipynb` -- when Python loops break JIT and how `lax.scan` saves you.